# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

### Finding 1: Engagement and Visibility Move Together (Finding #5)
> *"High scroll + high engagement = +11.2 health points. Visibility consistency compounds the effect."*

* **Methodology Question (Label Leakage):** Where does the `health_score` label come from? The methodology discloses that `health_score` is a composite metric explicitly calculated using `scroll depth (20 pts)` alongside impressions, position, and CTR. Claiming that scroll depth drives higher health scores represents circular target leakage, as the predictive feature is mathematically embedded in the ground-truth definition.
* **Methodology Question (Validation Design):** Does the validation design control for content lifecycle decay? Finding #2 demonstrates that older content naturally decays across all performance metrics. Without controlling for content age cohorts, the lower health score in low-engagement buckets may simply reflect decaying legacy pages rather than an isolated engagement failure.

---

### Finding 2: AI Traffic as a Different Signal (Finding #6)
> *"High-AI pages average ~9x more impressions (24.9K vs 2.7K) yet hold a weaker Google position (19.8 vs 14.2) than pages with no AI referrals."*

* **Methodology Question (Label Construction & Base Rate):** How was the `high_ai` label defined, and is the base rate sufficient to support a robust comparison? AI traffic represents only 1.06% of total portfolio sessions (17.3K of 1.6M), and the `high_ai` cohort contains only 873 pages out of 61.8K active records (~1.4%). With such extreme class rarity, aggregate metrics like average impressions are highly susceptible to skew from a handful of outlier pages.
* **Methodology Question (Validation & Group Isolation):** Does the validation isolate client-level clustering and survivor bias? The analysis relies on a pre-filtered active-content subset (impressions > 0 and sessions > 0). If the 873 `high_ai` pages are concentrated within a few specific enterprise domains, the observed impression gap may reflect client-level domain characteristics rather than a generalized search pattern.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My model under an honest split (before/after)

**Split Strategy & Before/After Comparison:**

To demonstrate validation leakage, I evaluated my Week-5 Logistic Regression model using two different splitting strategies.
*   **Before (Dishonest):** A standard 80/20 random split. This allows rows from the same `client_hash_id` to bleed into both the train and test sets, effectively allowing the model to memorize client-specific domain authority instead of learning the features.
*   **After (Honest):** An 80/20 `GroupShuffleSplit` grouped by `client_hash_id`. This guarantees the test set contains entirely unseen clients, providing a true measure of how the model generalizes.

The gap between these two AUC scores represents the amount of memorization occurring in the dishonest split.

In [7]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# 1. Connection Setup & Load Data (Full Month)
HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');")

FACT_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
DIM_PATH  = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

raw_df = con.execute(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.gsc_clicks,
        f.gsc_impressions,
        f.ga4_total_engagement_sec,
        c.word_count,
        c.backlinks,
        CASE
            WHEN (f.sessions_ai / (f.gsc_clicks + 1.0) > 0.35) AND (f.sessions_ai >= 5)
            THEN 1 ELSE 0
        END AS is_high_ai_spike
    FROM read_parquet('{FACT_PATH}') f
    JOIN read_parquet('{DIM_PATH}') c ON f.content_hash_id = c.content_hash_id
    WHERE f.gsc_data_available IS TRUE
      AND c.is_published IS TRUE
      AND c.is_deleted IS FALSE
""").df()

# 2. Content-Level Aggregation
frame = raw_df.groupby(["client_hash_id", "content_hash_id"]).agg(
    gsc_clicks=("gsc_clicks", "sum"),
    gsc_impressions=("gsc_impressions", "sum"),
    ga4_total_engagement_sec=("ga4_total_engagement_sec", "sum"),
    word_count=("word_count", "max"),
    backlinks=("backlinks", "max"),
    is_high_ai_spike=("is_high_ai_spike", "max")
).reset_index()

# 3. Clean Features
FEATURES = ["gsc_clicks", "gsc_impressions", "ga4_total_engagement_sec", "word_count", "backlinks"]
TARGET = "is_high_ai_spike"
frame[FEATURES] = frame[FEATURES].fillna(0)

X = frame[FEATURES]
y = frame[TARGET]
groups = frame['client_hash_id']

# ==========================================
# 4. BEFORE: The Dishonest (Random) Split
# ==========================================
X_train_rnd, X_test_rnd, y_train_rnd, y_test_rnd = train_test_split(X, y, test_size=0.20, random_state=42)

scaler_rnd = StandardScaler()
X_train_rnd_scaled = scaler_rnd.fit_transform(X_train_rnd)
X_test_rnd_scaled = scaler_rnd.transform(X_test_rnd)

model_rnd = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model_rnd.fit(X_train_rnd_scaled, y_train_rnd)

try:
    rnd_auc = roc_auc_score(y_test_rnd, model_rnd.predict_proba(X_test_rnd_scaled)[:, 1])
except ValueError:
    rnd_auc = float('nan')

# ==========================================
# 5. AFTER: The Honest (Grouped) Split
# ==========================================
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_grp, X_test_grp = X.iloc[train_idx], X.iloc[test_idx]
y_train_grp, y_test_grp = y.iloc[train_idx], y.iloc[test_idx]

scaler_grp = StandardScaler()
X_train_grp_scaled = scaler_grp.fit_transform(X_train_grp)
X_test_grp_scaled = scaler_grp.transform(X_test_grp)

model_grp = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model_grp.fit(X_train_grp_scaled, y_train_grp)

try:
    grp_auc = roc_auc_score(y_test_grp, model_grp.predict_proba(X_test_grp_scaled)[:, 1])
except ValueError:
    grp_auc = float('nan')

# 6. Output Comparison
print("=== Model Performance: Random vs. Grouped Split ===")
print(f"Dishonest (Random) Split AUC: {rnd_auc:.4f}")
print(f"Honest (Grouped) Split AUC:   {grp_auc:.4f}")
print(f"\nThe gap between these scores confirms that the random split allows my model to improperly memorize client-specific patterns.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Model Performance: Random vs. Grouped Split ===
Dishonest (Random) Split AUC: 0.8829
Honest (Grouped) Split AUC:   0.6864

The gap between these scores confirms that the random split allows my model to improperly memorize client-specific patterns.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

**The Leakage Trap Test**

Following the leakage taxonomy, my final feature set (`gsc_clicks`, `gsc_impressions`, `ga4_total_engagement_sec`, `word_count`, and `backlinks`) excludes label-derived features. My target, `is_high_ai_spike`, is computed directly from `sessions_ai` and `gsc_clicks`.

To verify my test harness is functioning and to demonstrate what a "label-derived feature" trap looks like, I will deliberately re-introduce `sessions_ai` into the model. If the harness works, the AUC should collapse toward 1.0, proving exactly why I must keep this feature excluded.

In [8]:
# 1. Fetch the leaky feature from the database
leaky_df = con.execute(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.sessions_ai
    FROM read_parquet('{FACT_PATH}') f
    JOIN read_parquet('{DIM_PATH}') c ON f.content_hash_id = c.content_hash_id
    WHERE f.gsc_data_available IS TRUE
      AND c.is_published IS TRUE
      AND c.is_deleted IS FALSE
""").df()

# Aggregate to the content level to match the main frame
leaky_agg = leaky_df.groupby(["client_hash_id", "content_hash_id"]).agg(
    sessions_ai=("sessions_ai", "max")
).reset_index()

# Merge into the existing frame from Section 2
trap_frame = pd.merge(frame, leaky_agg, on=["client_hash_id", "content_hash_id"])

# 2. Setup the leaky feature set AND FIX NANS
X_trap = trap_frame[FEATURES + ['sessions_ai']].copy()
X_trap = X_trap.fillna(0)

# Use the exact same grouped split indices from Section 2
X_train_trap, X_test_trap = X_trap.iloc[train_idx], X_trap.iloc[test_idx]

# Scale
scaler_trap = StandardScaler()
X_train_trap_scaled = scaler_trap.fit_transform(X_train_trap)
X_test_trap_scaled = scaler_trap.transform(X_test_trap)

# 3. Train and Score the Leaky Model
model_trap = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model_trap.fit(X_train_trap_scaled, y_train_grp)

trap_auc = roc_auc_score(y_test_grp, model_trap.predict_proba(X_test_trap_scaled)[:, 1])

# 4. Output the Trap Results
print("=== Leakage Trap Test ===")
print(f"Honest Grouped AUC (no leakage) : {grp_auc:.4f}")
print(f"Leaky Grouped AUC (+sessions_ai): {trap_auc:.4f}")
print("\nConclusion: The massive jump toward 1.0 confirms my test harness works and proves why I must keep 'sessions_ai' excluded.")

=== Leakage Trap Test ===
Honest Grouped AUC (no leakage) : 0.6864
Leaky Grouped AUC (+sessions_ai): 1.0000

Conclusion: The massive jump toward 1.0 confirms my test harness works and proves why I must keep 'sessions_ai' excluded.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

**Original Bold Claim (Causal & Overstated):**
"Increasing a page's word count and ignoring backlinks will guarantee a massive spike in AI traffic, because my model proves word count causes AI referrals."

**Rewritten Claim (Safe & Honest):**
"In the **observed** dataset, higher word counts and lower backlink profiles were **directionally** associated with an increased probability of an AI traffic spike. The **measured** feature weights do not prove causation, but they provide **decision-support** for prioritizing content depth over traditional link-building when targeting AI referral visibility."

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.